# Ressources critiques pour l’IA : raffinage et supply chain

Nettoyage minimal de fichiers minerais (format bruité) pour illustrer la dépendance géographique du raffinage et de la supply chain batteries.
Chemins : `../data/mineral/share-of-top-refining-country-for-20-energy-related-minerals.csv` et `../data/mineral/geographical-distribution-of-the-lfp-battery-supply-chain-2024.csv`.


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid")

root = Path('..') / 'data' / 'mineral'

# Fonction de parsing simple : insère des séparateurs puis extrait nom + deux chiffres
def parse_refining(path):
    txt = Path(path).read_text().replace('
','')
    txt = re.sub(r'([0-9])([A-Z])', r';', txt)
    matches = re.findall(r'([A-Za-z][A-Za-z \-]+);([0-9\.]+);([0-9\.]+)', txt)
    df = pd.DataFrame(matches, columns=['Mineral_or_country','Top_refiner_%','Second_%'])
    # Séparation grossière du pays collé au minéral
    df['Mineral'] = df['Mineral_or_country'].str.replace(r'^[A-Za-z]+(?=[A-Z])','', regex=True).str.strip()
    df.loc[df['Mineral']=='','Mineral'] = df['Mineral_or_country']
    df['Top_refiner_%'] = pd.to_numeric(df['Top_refiner_%'], errors='coerce')
    df['Second_%'] = pd.to_numeric(df['Second_%'], errors='coerce')
    return df

refining = parse_refining(root / 'share-of-top-refining-country-for-20-energy-related-minerals.csv')
refining.head()


In [ ]:
# Top 10 dépendances au raffineur principal
ax = sns.barplot(data=refining.sort_values('Top_refiner_%', ascending=False).head(10),
                 x='Top_refiner_%', y='Mineral', palette='rocket')
ax.set_title('Dépendance au principal pays raffineur')
ax.set_xlabel('Part du raffineur principal (%)')
plt.tight_layout(); plt.show()


In [ ]:
# Parsing supply chain LFP (4 régions principales)
def parse_lfp(path):
    txt = Path(path).read_text().replace('
','')
    txt = re.sub(r'([0-9])([A-Z])', r';', txt)
    matches = re.findall(r'([A-Za-z][A-Za-z \-]+);([0-9\.]+);([0-9\.]+);([0-9\.]+);([0-9\.]+)', txt)
    df = pd.DataFrame(matches, columns=['Segment','China','Europe','United States','Japan_Korea'])
    for c in ['China','Europe','United States','Japan_Korea']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

lfp = parse_lfp(root / 'geographical-distribution-of-the-lfp-battery-supply-chain-2024.csv')
lfp.head()


In [ ]:
# Visualisation par segment
lfp_melt = lfp.melt(id_vars='Segment', var_name='Region', value_name='Share_%')
ax = sns.barplot(data=lfp_melt, x='Share_%', y='Segment', hue='Region', palette='Set2')
ax.set_title('Chaîne LFP : répartition géographique (%)')
plt.tight_layout(); plt.show()


## Lecture rapide
- Raffinage : plusieurs minerais critiques sont ultra-concentrés (>90%) chez un seul pays, créant un risque de supply chain pour le hardware IA.
- Chaîne LFP : la Chine domine largement la plupart des segments, l’Europe/US restent marginales.
- Les fichiers sources sont bruités ; le parsing est simplifié mais suffisant pour visualiser la dépendance.
